# Solenoids - Analytical Calculation


In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Physical constant
mu0 = 4 * np.pi * 1e-7  # H/m

def calculate_Bz(r, z, J, L, Ri, Rf):
    """
    Computes the axial magnetic field Bz at position (r, z) for a finite solenoid.
    
    Parameters:
        r  : radial position (m)
        z  : axial position (m)
        J  : current density (A/m^2)
        L  : solenoid length (m)
        Ri : inner radius (m)
        Rf : outer radius (m)
    """
    Bz = np.zeros_like(z)
    
    for i, z_val in enumerate(z):
        term1 = (z_val + L/2) * np.log(
            (np.sqrt(Rf**2 + (z_val + L/2)**2) + Rf) /
            (np.sqrt(Ri**2 + (z_val + L/2)**2) + Ri)
        )
        term2 = (z_val - L/2) * np.log(
            (np.sqrt(Rf**2 + (z_val - L/2)**2) + Rf) /
            (np.sqrt(Ri**2 + (z_val - L/2)**2) + Ri)
        )
        Bz[i] = (mu0 * J / 2) * (term1 - term2)
    
    return Bz

def calculate_Em(L, Th, Rm, Ri, Rf):
    """
    Calculate the magnetic energy Em.

    Parameters:
    L (float): Solenoid length.
    Th (float): Thickness of the solenoid.
    Rm (float): Mean radius.
    eta (float): Geometric factor.
    em (float) : Magnetic energy density (em = Em/Volume).

    Returns:
    float: Magnetic energy Em.
    """
    Th = Rf - Ri
    Rm = Ri + Th / 2
    eta = 0.2235*(L + Th)
    mu0 = 4 * np.pi * 1e-7  # Permeability of free space
    Em = (J**2 * mu0 * Rm * (L * Th)**2 / 2) * (np.log(8 * Rm / eta) * (1 + 3 * eta**2 / (16 * Rm**2)) - (2 + eta**2 / (16 * Rm**2)))
    return Em

def calculate_stress(Ri, Rf, L, J, nu, B1, B2):
    """
    Calculate the radial stress σr and hoop stress σθ.

    Parameters:
    Ri (float): Bore radius.
    Rf (float): External radius.
    L (float): Solenoid length.
    J (float): Current density.
    nu (float): Poisson's ratio.
    B1 (float): Longitudinal field at r = Ri (can be analitically estimated).
    B2 (float): Longitudinal field at r = Rf (no mathematical expression).
    kappa (float): Ratio of B2 to B1 (in our case can vary from -0.02 and -0.9 TBC).

    Returns:
    tuple: Radial stress σr and hoop stress σθ.
    """
    alpha = Rf / Ri
    kappa = B2 / B1
    rho = np.linspace(Ri, Rf, 100) / Ri

    sigma_r = (J**2 * B1 * Ri**2 / (alpha - 1)) * (
        (2 + nu) / 3 * (alpha - kappa) * (alpha**2 + alpha + 1 - alpha**2 / rho**2) / (alpha + 1 - rho) -
        (3 + nu) / 8 * (1 - kappa) * (alpha**2 + 1 - alpha**2 / rho**2 - rho**2)
    )

    sigma_theta = (J**2 * B1 * Ri**2 / (alpha - 1)) * (
        (alpha - kappa) * ((2 + nu) / 3 * (alpha**2 + alpha + 1 + alpha**2 / rho**2) / (alpha + 1) - (1 + 2 * nu) / 3 / rho) -
        (1 - kappa) * ((3 + nu) / 8 * (alpha**2 + 1 + alpha**2 / rho**2) - (1 + 3 * nu) / 8 / rho**2)
    )

    return sigma_r, sigma_theta